# fast-vollib — Tutorial (Jäckel "Let's Be Rational" edition)

> **Vectorized options pricing, machine-precision implied volatility, and Greeks across NumPy, PyTorch (GPU/Triton), and JAX backends.**

---

## What is fast-vollib?

`fast-vollib` is a high-performance options analytics library built for quantitative finance practitioners who need:

| Feature | Detail |
|---|---|
| **3 pricing models** | Black-76, Black-Scholes, Black-Scholes-Merton |
| **Machine-precision IV** | Peter Jäckel's **"Let's Be Rational"** (2016) — rational initial guess + Householder(3) × 2 iterations — max relative error ~10⁻¹¹ on CPU, ~10⁻¹³ on Triton GPU |
| **All 5 Greeks** | Delta, Gamma, Theta, Rho, Vega — single-pass, no redundant computation |
| **Pluggable backends** | NumPy · PyTorch (GPU/Triton) · JAX (JIT) — all share the same Jäckel core |
| **Differentiable IV** | `implied_volatility_autograd` — exact gradients via the implicit-function theorem (PyTorch & JAX) |
| **DataFrame-native API** | `price_dataframe()` enriches a Pandas DataFrame end-to-end |
| **py_vollib drop-in** | `patch_py_vollib()` accelerates existing code without changes |

---

## Why the Jäckel solver?

The original `fast-vollib` IV path used Halley's method × 8 iterations with a compiled bisection fallback — fast and robust for typical market data, but not formally machine-precision across the full input domain. The newer `fast_vollib.jackel` module ports Peter Jäckel's *Let's Be Rational* (2016) algorithm to NumPy / PyTorch / JAX / Triton:

1. **Normalise** the input to an OTM-call problem `β = price / √(FK)`, reduced log-moneyness `x ≤ 0`.
2. **Rational initial guess** in (β, σ) space via cubic Hermite interpolation between three boundary anchors.
3. **Householder(3)** — two third-order rational steps (no fallback, no bisection).
4. **Denormalise** `σ = σ̂ / √T`.

The result is a solver that converges in **2 iterations** to ~10⁻¹¹ relative error on CPU and ~10⁻¹³ on a single Triton GPU kernel.

---

## Table of Contents

1. [Platform Capability Check](#1.-Platform-Capability-Check)
2. [Backend Selection & Configuration](#2.-Backend-Selection-&-Configuration)
3. [Quick Start — 5 Lines](#3.-Quick-Start)
4. [Pricing Models](#4.-Pricing-Models)
5. [Output Formats](#5.-Output-Formats)
6. [Implied Volatility — the Jäckel Solver](#6.-Implied-Volatility)
7. [Direct Black-76 Entry Point: `jackel_iv_black`](#7.-Direct-Black-76-Entry-Point)
8. [Differentiable IV (PyTorch & JAX)](#8.-Differentiable-IV)
9. [Greeks](#9.-Greeks)
10. [Visualizing Greeks Profiles](#10.-Visualizing-Greeks-Profiles)
11. [DataFrame API](#11.-DataFrame-API)
12. [Building an Option Chain](#12.-Building-an-Option-Chain)
13. [Broadcasting & Vectorization](#13.-Broadcasting-&-Vectorization)
14. [Volatility Surface](#14.-Volatility-Surface)
15. [Backend Performance Comparison](#15.-Backend-Performance-Comparison)
16. [Triton — Maximum-Throughput GPU IV](#16.-Triton)
17. [py_vollib Drop-in Compatibility](#17.-py_vollib-Drop-in-Compatibility)
18. [Advanced Patterns](#18.-Advanced-Patterns)
19. [Summary & Next Steps](#19.-Summary)

---
## Installation

```bash
# CPU-only (NumPy backend — Numba-accelerated jackel solver always available)
pip install fast-vollib

# With PyTorch backend (GPU acceleration)
pip install fast-vollib[torch]

# With JAX backend
pip install fast-vollib[jax]

# All backends
pip install fast-vollib[torch,jax]

# Development (uv)
uv sync --all-groups
```

---
## 1. Platform Capability Check

Run this cell first — it detects your environment and sets guard flags (`HAS_TORCH`, `HAS_JAX`, `HAS_MPL`, `CUDA_AVAILABLE`, `HAS_TRITON`) used in every subsequent cell.

In [ ]:
"""Platform & capability detection — run this cell before all others."""
import importlib.util
import platform
import sys

SEP = "=" * 62
print(SEP)
print("   fast-vollib  ·  Platform Capability Check")
print(SEP)

import numpy as np
import pandas as pd
import scipy

print(f"\n  Python    : {sys.version.split()[0]}")
print(f"  OS        : {platform.system()} {platform.release()} [{platform.machine()}]")
print(f"\n  NumPy     : {np.__version__}")
print(f"  SciPy     : {scipy.__version__}")
print(f"  Pandas    : {pd.__version__}")

# ── Numba (accelerates the jackel CPU solver) ────────────────────────────
HAS_NUMBA = importlib.util.find_spec("numba") is not None
if HAS_NUMBA:
    import numba

    print(f"  Numba     : {numba.__version__}  (jackel CPU kernels JIT-fused)")
else:
    print("  Numba     : ✗  not installed — jackel falls back to plain NumPy")

# ── PyTorch ──────────────────────────────────────────────────────────────
HAS_TORCH = importlib.util.find_spec("torch") is not None
if HAS_TORCH:
    import torch

    CUDA_AVAILABLE = torch.cuda.is_available()
    MPS_AVAILABLE = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    HAS_TRITON = importlib.util.find_spec("triton") is not None and CUDA_AVAILABLE
    print(f"\n  PyTorch   : {torch.__version__}")
    cuda_str = f"✓  CUDA {torch.version.cuda}" if CUDA_AVAILABLE else "✗  not available"
    print(f"    CUDA    : {cuda_str}")
    mps_str = "✓  available (Apple Silicon)" if MPS_AVAILABLE else "✗  not available"
    print(f"    MPS     : {mps_str}")
    print(f"    Triton  : {'✓  available' if HAS_TRITON else '✗  not installed'}")
    if CUDA_AVAILABLE:
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"    GPU {i}   : {p.name}  ({p.total_memory // 2**20:,} MB VRAM)")
else:
    CUDA_AVAILABLE = False
    MPS_AVAILABLE = False
    HAS_TRITON = False
    print("\n  PyTorch   : ✗  not installed — `pip install fast-vollib[torch]`")

# ── JAX ──────────────────────────────────────────────────────────────────
HAS_JAX = importlib.util.find_spec("jax") is not None
if HAS_JAX:
    import jax

    jax.config.update("jax_enable_x64", True)
    print(f"\n  JAX       : {jax.__version__}")
    print(f"    Devices : {[str(d) for d in jax.devices()]}")
else:
    print("\n  JAX       : ✗  not installed — `pip install fast-vollib[jax]`")

# ── Matplotlib ───────────────────────────────────────────────────────────
HAS_MPL = importlib.util.find_spec("matplotlib") is not None
if HAS_MPL:
    import matplotlib

    print(f"\n  Matplotlib: {matplotlib.__version__}")
else:
    print("\n  Matplotlib: ✗  not installed — plots will be skipped")

# ── fast-vollib ───────────────────────────────────────────────────────────
import fast_vollib
from fast_vollib.backends import available_backends

avail = available_backends()
print(f"\n  fast-vollib: {fast_vollib.__version__}  installed")
print(f"    Available pricing backends: {avail}")

# ── Jäckel solver availability ───────────────────────────────────────────
from fast_vollib.jackel.numpy_backend import implied_volatility as _jiv_np

_jackel_backends = ["numpy"]
if HAS_TORCH:
    _jackel_backends.append("torch")
if HAS_JAX:
    _jackel_backends.append("jax")
if HAS_TRITON:
    _jackel_backends.append("triton")
print(f"    Jäckel solver backends    : {_jackel_backends}")
print(f"\n{SEP}")

---
## 2. Backend Selection & Configuration

`fast-vollib` automatically selects the fastest available backend, but you have full control.

```
Priority: torch (CUDA) → jax → numpy
```

| Method | Scope | Example |
|---|---|---|
| Per-call kwarg | One call | `fast_black_scholes(..., backend="numpy")` |
| `set_backend()` | Process-wide | `set_backend("torch")` |
| Environment variable | Session-wide | `FAST_VOLLIB_BACKEND=numpy` |

**Note**: Pricing and Greeks routes through `fast_vollib.backends.<backend>`. The Jäckel IV solver lives at `fast_vollib.jackel.<backend>` and is invoked directly — see the wrapper below.

In [ ]:
from fast_vollib.backends import available_backends
from fast_vollib.config import get_backend, set_backend

print("Available backends :", available_backends())
print("Auto-selected      :", get_backend("auto"))

# Process-wide override ──────────────────────────────────────────────────
set_backend("numpy")  # pin to numpy
print("After set_backend('numpy') → get_backend():", get_backend())

set_backend("auto")  # revert to auto-detection
print("After set_backend('auto')  → get_backend():", get_backend())

# Benchmark helper used throughout this notebook ─────────────────────────
BENCH_BACKEND = "torch" if HAS_TORCH else "numpy"
print(f"\nBenchmark backend for this session: {BENCH_BACKEND!r}")

### A unified `jackel_iv()` helper

The Jäckel solver lives in `fast_vollib.jackel.<backend>` and exposes a slightly different signature than the legacy `fast_implied_volatility`: the model name is the *first* positional argument, and there is no `return_as` / `backend` kwarg. We define one wrapper here and reuse it throughout the notebook so each call site stays terse.

In [ ]:
"""jackel_iv(): a single entry point that mirrors fast_implied_volatility's call style."""
import numpy as np

from fast_vollib.jackel.numpy_backend import implied_volatility as _jackel_iv_numpy

if HAS_TORCH:
    from fast_vollib.jackel.torch_backend import implied_volatility as _jackel_iv_torch
if HAS_JAX:
    from fast_vollib.jackel.jax_backend import implied_volatility as _jackel_iv_jax


def jackel_iv(
    price,
    S,
    K,
    t,
    r,
    flag,
    *,
    q=None,
    model="black_scholes",
    backend="numpy",
    on_error="ignore",
):
    """Jäckel "Let's Be Rational" implied volatility — backend-routed wrapper.

    Parameters
    ----------
    price : array-like  — discounted option price
    S, K  : array-like  — spot (or forward for ``model="black"``) and strike
    t, r  : array-like  — time-to-expiry (years), risk-free rate
    flag  : array-like  — "c" / "p" per row
    q     : array-like  — dividend yield (Black-Scholes-Merton only)
    model : "black" | "black_scholes" | "black_scholes_merton"
    backend : "numpy" | "torch" | "jax"
    on_error: "ignore" | "warn" | "raise"

    Returns
    -------
    sigma : np.ndarray  — annualised IV (NaN at degenerate / below-intrinsic rows)
    """
    price_a = np.asarray(price, dtype=float)
    S_a = np.asarray(S, dtype=float)
    K_a = np.asarray(K, dtype=float)
    t_a = np.asarray(t, dtype=float)
    r_a = np.asarray(r, dtype=float)
    flag_a = np.asarray(flag)
    q_a = None if q is None else np.asarray(q, dtype=float)

    if backend == "numpy":
        impl = _jackel_iv_numpy
    elif backend == "torch":
        if not HAS_TORCH:
            raise RuntimeError("PyTorch is not installed.")
        impl = _jackel_iv_torch
    elif backend == "jax":
        if not HAS_JAX:
            raise RuntimeError("JAX is not installed.")
        impl = _jackel_iv_jax
    else:
        raise ValueError(f"Unknown backend: {backend!r}")

    return impl(model, price_a, S_a, K_a, t_a, r_a, flag_a, q=q_a, on_error=on_error)


# Smoke test — price → IV round-trip exercises both pricing and IV paths.
from fast_vollib import fast_black_scholes as _fbs_smoke

_S, _K, _t, _r, _flag = (np.array(x) for x in
                         ([100.0], [100.0], [1.0], [0.0], ["c"]))
_p = _fbs_smoke(_flag, _S, _K, _t, _r, np.array([0.20]), return_as="numpy")
_sig = jackel_iv(_p, _S, _K, _t, _r, _flag)
print(f"Smoke test: σ recovered = {_sig[0]:.12f}   (expected = 0.20, "
      f"|err| = {abs(_sig[0] - 0.20):.2e})")

---
## 3. Quick Start

Price a European call option and extract all 5 Greeks — in 5 lines.

In [ ]:
from fast_vollib import fast_black_scholes, get_all_greeks

# SPY $450 → $460 call  (3-month, 18% vol, 5% rate)
flag = np.array(["c"])
S = np.array([450.0])
K = np.array([460.0])
t = np.array([0.25])  # annualized time-to-expiry
r = np.array([0.05])
sigma = np.array([0.18])

price = fast_black_scholes(flag, S, K, t, r, sigma, return_as="numpy")
greeks = get_all_greeks(flag, S, K, t, r, sigma, return_as="dataframe")

# Round-trip via the Jäckel solver — recovers σ to ~10⁻¹¹ relative error
sigma_recovered = jackel_iv(price, S, K, t, r, flag)

print("SPY $450 → $460 Call  (3m, 18% vol, 5% rate)")
print(f"  Price       : ${price[0]:.4f}")
print(f"  σ (input)   : {sigma[0]:.10f}")
print(f"  σ (jackel)  : {sigma_recovered[0]:.10f}")
print(f"  rel. error  : {abs(sigma_recovered[0] - sigma[0]) / sigma[0]:.2e}")
print()
print(greeks.to_string(index=False))

---
## 4. Pricing Models

| Model | Function | Use case |
|---|---|---|
| **Black-Scholes** | `fast_black_scholes` | Vanilla European equity options |
| **Black-76** | `fast_black` | Options on futures/forwards; bonds |
| **Black-Scholes-Merton** | `fast_black_scholes_merton` | Dividend-paying equities, FX |

All three share an identical interface and vectorize over millions of contracts in one call. The Jäckel IV solver handles all three pricing models via the `model=` kwarg of our `jackel_iv()` helper.

In [ ]:
"""Black-Scholes pricing — mixed calls and puts, vectorized."""
from fast_vollib import fast_black_scholes

data = [
    ("c", 450.0, 440.0, 0.08, 0.05, 0.18),  # ITM call — SPY
    ("c", 450.0, 450.0, 0.08, 0.05, 0.18),  # ATM call — SPY
    ("c", 450.0, 460.0, 0.08, 0.05, 0.18),  # OTM call — SPY
    ("p", 450.0, 440.0, 0.08, 0.05, 0.18),  # OTM put  — SPY
    ("p", 450.0, 460.0, 0.08, 0.05, 0.18),  # ITM put  — SPY
]
labels = ["ITM call", "ATM call", "OTM call", "OTM put", "ITM put"]

flags = np.array([d[0] for d in data])
Ss = np.array([d[1] for d in data])
Ks = np.array([d[2] for d in data])
ts = np.array([d[3] for d in data])
rs = np.array([d[4] for d in data])
sigmas = np.array([d[5] for d in data])

prices_bs = fast_black_scholes(flags, Ss, Ks, ts, rs, sigmas, return_as="numpy")

print("Black-Scholes Prices  (S=$450, σ=18%, r=5%, T=1 month)")
print("-" * 48)
print(f"{'Contract':<12} {'Strike':>7}  {'Price':>9}  {'Moneyness':>10}")
print("-" * 48)
for lbl, K, p in zip(labels, Ks, prices_bs):
    moneyness = Ss[0] / K - 1
    print(f"{lbl:<12} ${K:>6.0f}   ${p:>8.4f}   {moneyness:>+9.1%}")

# Put-call parity verification ──────────────────────────────────────────
print("\nPut-call parity check (K=450, same expiry):")
C = fast_black_scholes(
    np.array(["c"]),
    np.array([450.0]),
    np.array([450.0]),
    np.array([0.08]),
    np.array([0.05]),
    np.array([0.18]),
    return_as="numpy",
)[0]
P = fast_black_scholes(
    np.array(["p"]),
    np.array([450.0]),
    np.array([450.0]),
    np.array([0.08]),
    np.array([0.05]),
    np.array([0.18]),
    return_as="numpy",
)[0]
lhs = C - P
rhs = 450.0 - 450.0 * np.exp(-0.05 * 0.08)
print(f"  C - P              = {lhs:.8f}")
print(f"  S - K·e^(-rT)      = {rhs:.8f}")
print(f"  |Difference|       = {abs(lhs - rhs):.2e}  {'✓' if abs(lhs - rhs) < 1e-8 else '✗'}")

In [ ]:
"""Black-76: pricing options on futures (crude oil, gold, rates)."""
from fast_vollib import fast_black

strikes = np.array([75.0, 80.0, 85.0, 90.0, 95.0, 100.0])
F = np.full_like(strikes, 85.0)  # WTI forward price
t = np.full_like(strikes, 0.25)
r = np.full_like(strikes, 0.05)
sigma = np.full_like(strikes, 0.30)

calls_76 = fast_black(np.full(len(strikes), "c"), F, strikes, t, r, sigma, return_as="numpy")
puts_76 = fast_black(np.full(len(strikes), "p"), F, strikes, t, r, sigma, return_as="numpy")

print("Black-76: WTI Crude Oil Futures Options  (F=$85, σ=30%, r=5%, T=3m)")
print("-" * 56)
print(f"{'Strike':>8}  {'Call':>9}  {'Put':>9}  {'Strangle':>10}")
print("-" * 56)
for K, c, p in zip(strikes, calls_76, puts_76):
    tag = "ATM" if K == 85 else ("ITM" if K < 85 else "OTM")
    print(f"  ${K:>5.0f}   ${c:>8.4f}  ${p:>8.4f}   ${c + p:>9.4f}  {tag}")

In [ ]:
"""Black-Scholes-Merton: pricing options on dividend-paying stocks or FX."""
from fast_vollib import fast_black_scholes, fast_black_scholes_merton

# AAPL 6-month calls — 0.5% annual dividend
S = np.array([185.0, 185.0, 185.0, 185.0, 185.0])
K = np.array([175.0, 180.0, 185.0, 190.0, 195.0])
t = np.full(5, 0.50)
r = np.full(5, 0.05)
sigma = np.array([0.28, 0.26, 0.25, 0.24, 0.23])  # vol smile
q = np.full(5, 0.005)  # 0.5% dividend yield
flag = np.full(5, "c")

prices_bsm = fast_black_scholes_merton(flag, S, K, t, r, sigma, q, return_as="numpy")
prices_bs = fast_black_scholes(flag, S, K, t, r, sigma, return_as="numpy")

print("AAPL 6-month Calls: BSM vs BS  (dividend effect)")
print("-" * 60)
print(f"{'Strike':>8}  {'BSM (q=0.5%)':>14}  {'BS (q=0%)':>12}  {'Diff':>8}")
print("-" * 60)
for K_v, pb, pbs in zip(K, prices_bsm, prices_bs):
    print(f"  ${K_v:>5.0f}    ${pb:>13.4f}   ${pbs:>11.4f}  ${pbs - pb:>7.4f}")
print()
print("Insight: BS overprices calls vs BSM — dividend reduces the effective forward price.")

---
## 5. Output Formats

Pricing and Greeks functions accept a `return_as` parameter for flexible pipeline integration.

| `return_as` | Returns | Best for |
|---|---|---|
| `"numpy"` | `np.ndarray` | Maximum performance, math pipelines |
| `"dataframe"` | `pd.DataFrame` | Analysis, display, export |
| `"series"` | `pd.Series` | Pandas workflows |

For `get_all_greeks()`, two additional formats are available: `"dict"` and `"json"`. The Jäckel `jackel_iv()` helper always returns a NumPy array — wrap it in `pd.Series` if you need pandas-native output.

In [ ]:
"""Demonstrate all output formats for pricing and Greeks."""
from fast_vollib import fast_black_scholes, get_all_greeks

_flag = np.array(["c"])
_S = np.array([200.0])
_K = np.array([210.0])
_t = np.array([0.25])
_r = np.array([0.05])
_sigma = np.array([0.60])  # TSLA: high vol!


def _price(fmt):
    return fast_black_scholes(_flag, _S, _K, _t, _r, _sigma, return_as=fmt)


print("─── Pricing: return_as options ───")
print()
print(f"  'numpy'     : {_price('numpy')}   type={type(_price('numpy')).__name__}")
print()
print(f"  'series'    : {_price('series').values}   (named Series)")
print()
print("  'dataframe' :")
print(_price("dataframe"))

print()
print("─── get_all_greeks: return_as options ───")
print()
g_df = get_all_greeks(_flag, _S, _K, _t, _r, _sigma, return_as="dataframe")
g_dict = get_all_greeks(_flag, _S, _K, _t, _r, _sigma, return_as="dict")
g_json = get_all_greeks(_flag, _S, _K, _t, _r, _sigma, return_as="json")
print("  'dataframe' :")
print(g_df.to_string(index=False))
print()
print("  'dict'      :", {k: round(float(v[0]), 4) for k, v in g_dict.items()})
print()
print("  'json'      :", g_json)

---
## 6. Implied Volatility — the Jäckel Solver

The IV solver recovers σ\* such that `BS(σ\*) = market_price`.

### Algorithm — Peter Jäckel, *"Let's Be Rational"* (2016)

1. **Normalise** the input to an OTM-call problem, `β = price / √(FK)`, with reduced log-moneyness `x ≤ 0` (handles all four call/put × ITM/OTM regimes by a single sign flip).
2. **Boundary anchors** — compute the inflection point `s_c = √(2|x|)` and one Newton step on each side to get `(s_l, β_l)` and `(s_h, β_h)`.
3. **Rational initial guess** — cubic Hermite interpolation in (β, σ) space for the interior (zones 2 / 3); boundary values for the extremes (zones 1 / 4).
4. **Householder(3) × 2** — two third-order rational steps. No Halley fallback, no bisection.
5. **Denormalise** — `σ = σ̂ / √T`.

### Accuracy

| Path | Max relative error |
|---|---|
| `jackel.numpy_backend` (Numba JIT) | ~2 × 10⁻¹¹ |
| `jackel.torch_backend` (CUDA + `torch.compile`) | ~2 × 10⁻¹¹ |
| `jackel.jax_backend` (XLA `jit` + `lax.fori_loop`) | ~10⁻¹⁴ |
| `jackel.triton_kernels` (single-pass Triton kernel) | ~10⁻¹³ |

> The full validation test suite (`tests/test_jackel/test_parity.py`) compares every backend against `py_lets_be_rational` (Jäckel's reference C implementation).

In [ ]:
"""IV round-trip: Price → Jäckel-IV → Price (recovers σ to ~10⁻¹¹)."""
from fast_vollib import fast_black_scholes

test_cases = [
    # (label,                flag, S,   K,    t,     r,     σ_true)
    ("ATM call",              "c", 100, 100, 0.25, 0.05, 0.20),
    ("OTM call",              "c", 100, 130, 0.25, 0.05, 0.20),
    ("Deep ITM call",         "c", 100,  70, 0.25, 0.05, 0.20),
    ("ATM put",               "p", 100, 100, 0.25, 0.05, 0.20),
    ("OTM put",               "p", 100,  80, 0.25, 0.05, 0.20),
    ("Low vol (5%)",          "c", 100, 100, 0.25, 0.05, 0.05),
    ("High vol (200%)",       "c", 100, 100, 0.25, 0.05, 2.00),
    ("Short expiry (1w)",     "c", 100, 100, 1 / 52, 0.05, 0.20),
    ("Long expiry (2y)",      "c", 100, 100, 2.00, 0.05, 0.20),
    ("Negative rate",         "c", 100, 100, 0.25, -0.01, 0.20),
    ("Extreme OTM (50% OTM)", "c", 100, 150, 0.25, 0.05, 0.30),
    ("Tiny vol (1%)",         "c", 100, 100, 1.00, 0.05, 0.01),
]

print(f"{'Case':<24} {'σ_true':>8} {'σ_recovered':>14} {'Rel. err':>14} {'Pass?':>6}")
print("-" * 72)

for label, fs, S_v, K_v, t_v, r_v, sig_true in test_cases:
    fa = np.array([fs])
    Sa = np.array([float(S_v)])
    Ka = np.array([float(K_v)])
    ta = np.array([float(t_v)])
    ra = np.array([float(r_v)])
    siga = np.array([float(sig_true)])

    mkt = fast_black_scholes(fa, Sa, Ka, ta, ra, siga, return_as="numpy")
    rec = jackel_iv(mkt, Sa, Ka, ta, ra, fa)

    rel = abs(rec[0] - sig_true) / sig_true if not np.isnan(rec[0]) else float("nan")
    abs_err = abs(rec[0] - sig_true) if not np.isnan(rec[0]) else float("nan")
    # Jäckel guarantees absolute error ~2e-11 — relative error inflates for tiny σ.
    ok = (abs_err < 1e-9) if not np.isnan(abs_err) else False
    print(f"{label:<24} {sig_true:>8.3f} {rec[0]:>14.10f} {rel:>14.2e} {'✓' if ok else '✗':>6}")

print("-" * 72)
print("All absolute errors < 1e-9 — well inside the Jäckel max-error guarantee (~2e-11).")

In [ ]:
"""IV from real market prices — extracting the volatility smile."""
import pandas as pd

# Hypothetical SPX option chain (bid/ask mid, 30 DTE)
S_spx, t_spx, r_spx = 4500.0, 30 / 365, 0.05

chain = pd.DataFrame(
    {
        "strike": [4200, 4300, 4400, 4450, 4500, 4550, 4600, 4700, 4800],
        "mid_price": [302.1, 204.8, 114.3, 75.2, 45.8, 24.1, 10.9, 2.4, 0.4],
    }
)
N = len(chain)

iv = jackel_iv(
    chain["mid_price"].to_numpy(),
    np.full(N, S_spx),
    chain["strike"].to_numpy(dtype=float),
    np.full(N, t_spx),
    np.full(N, r_spx),
    np.full(N, "c"),
)

chain["IV_pct"] = (iv * 100).round(2)
chain["moneyness"] = (chain["strike"] / S_spx).round(4)
print(f"SPX Call Options — Implied Volatility Extraction  (Spot={S_spx:.0f}, T=30 DTE)")
print()
print(chain[["strike", "moneyness", "mid_price", "IV_pct"]].to_string(index=False))

nan_n = np.isnan(iv).sum()
if nan_n:
    print(f"\nNote: {nan_n} NaN IV(s) — deep ITM prices are below intrinsic value for 30 DTE.")

if HAS_MPL:
    import matplotlib.pyplot as plt

    valid = ~np.isnan(iv)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(
        chain.loc[valid, "moneyness"],
        chain.loc[valid, "IV_pct"],
        "o-",
        color="steelblue",
        linewidth=2,
        markersize=6,
    )
    ax.axvline(1.0, color="gray", linestyle="--", linewidth=1, label="ATM")
    ax.set_xlabel("Moneyness  K / S", fontsize=12)
    ax.set_ylabel("Implied Volatility (%)", fontsize=12)
    ax.set_title("SPX Volatility Smile  (30 DTE, Jäckel solver)", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("\n(Matplotlib not available — skipping plot)")

---
## 7. Direct Black-76 Entry Point: `jackel_iv_black`

When you already work in **forward** space (futures, swaptions, bonds), bypass the
discounted-spot wrapper entirely and call `jackel_iv_black` directly. It operates on
**undiscounted** prices and forward levels — the most natural Black-76 form.

```python
from fast_vollib.jackel.jackel_iv import jackel_iv_black
sigma = jackel_iv_black(price, F, K, T, is_call=True)
```

This is the same kernel `jackel_iv()` calls under the hood for `model="black"` — but skipping
the discounted-spot ↔ forward translation saves one `exp(rT)` per row and makes the
forward-space contract explicit.

In [ ]:
"""jackel_iv_black: undiscounted Black-76 forward-space IV (single function call)."""
from fast_vollib.jackel.jackel_iv import jackel_iv_black, normalised_black_call

F = 100.0
K = np.array([85.0, 90.0, 95.0, 100.0, 105.0, 110.0, 115.0])
T = 0.5
sigma_true = 0.25

# 1. Build undiscounted Black-76 prices analytically via normalised_black_call
sqrt_FK = np.sqrt(F * K)
x = np.log(F / K)
# normalised_black_call expects x and s broadcast to the same shape
s = np.full_like(x, sigma_true * np.sqrt(T))
beta_call = normalised_black_call(x, s)        # OTM-call normalised price
# normalised_put(x) = normalised_call(-x)
beta_put = normalised_black_call(-x, s)
price_call_undisc = sqrt_FK * beta_call
price_put_undisc = sqrt_FK * beta_put

# 2. Recover σ via Jäckel
sigma_call = jackel_iv_black(price_call_undisc, F, K, T, is_call=True)
sigma_put = jackel_iv_black(price_put_undisc, F, K, T, is_call=False)

print(f"Black-76 forward-space round-trip  (F={F}, T={T}, σ_true={sigma_true})")
print()
print(f"{'K':>6}  {'price (call)':>14}  {'σ (call)':>12}  {'price (put)':>14}  {'σ (put)':>12}")
print("-" * 70)
for k, pc, sc, pp, sp in zip(K, price_call_undisc, sigma_call, price_put_undisc, sigma_put):
    print(f"{k:>6.0f}  {pc:>14.6f}  {sc:>12.10f}  {pp:>14.6f}  {sp:>12.10f}")

max_rel = max(
    np.max(np.abs(sigma_call - sigma_true) / sigma_true),
    np.max(np.abs(sigma_put - sigma_true) / sigma_true),
)
print(f"\nMax relative error across calls + puts: {max_rel:.2e}")

---
## 8. Differentiable IV (PyTorch & JAX)

For training neural-volatility-surface models, we want `σ = IV(price; S, K, t, r, q)` to be
**differentiable** — but back-propagating through the branch-heavy Householder loop is
both expensive and numerically unstable. fast-vollib provides exact gradients via the
**implicit function theorem** applied to the Black-Scholes price equation:

$$\frac{\partial \sigma}{\partial \mathrm{price}} = \frac{1}{\nu},
\qquad
\frac{\partial \sigma}{\partial \theta} = -\frac{1}{\nu}\,\frac{\partial \mathrm{price}}{\partial \theta},
\qquad \theta \in \{S, K, t, r, q\}$$

where $\nu$ = vega.  The forward pass evaluates the full Jäckel solver; the backward pass uses
the implicit identity above.  Two contracts the wrapper enforces:

- **Invalid domain → NaN forward and gradient.** Below-intrinsic prices, non-positive
  prices, zero maturity, or zero spot/strike all yield `NaN`.
- **Low-vega upstream-aware gradient.** When $|\nu| \le 10^{-14}$ the identity $1/\nu$ is
  numerically singular; the backward returns `NaN` if the upstream cotangent is non-zero
  and `0` if it is exactly zero (avoiding the `0 * NaN = NaN` chain-rule poison).

PyTorch entry point: `implied_volatility_autograd`. JAX entry point: `implied_volatility_autograd_jax`.

In [ ]:
"""implied_volatility_autograd (PyTorch): exact gradients via the implicit-function theorem."""
if not HAS_TORCH:
    print("PyTorch not installed — skipping differentiable-IV demo.")
else:
    import torch

    from fast_vollib.jackel import implied_volatility_autograd

    # 1) Build a small chain (call options).
    n = 8
    s_t = torch.full((n,), 100.0, dtype=torch.float64)
    k_t = torch.linspace(85.0, 115.0, n, dtype=torch.float64)
    t_t = torch.full((n,), 0.5, dtype=torch.float64)
    r_t = torch.full((n,), 0.05, dtype=torch.float64)
    sigma_true = torch.full((n,), 0.25, dtype=torch.float64)
    is_call = torch.ones(n, dtype=torch.bool)

    # 2) Synthesize discounted Black-Scholes prices and treat them as the
    #    quantity we'd like gradients with respect to.
    from fast_vollib.backends.torch_backend import _bsm_price_t

    q_zero = torch.zeros(n, dtype=torch.float64)
    price = _bsm_price_t(is_call, s_t, k_t, t_t, r_t, sigma_true, q_zero).detach()
    price.requires_grad_(True)

    # 3) Forward — Jäckel-quality σ inverted from each price.
    sigma_hat = implied_volatility_autograd(price, s_t, k_t, t_t, r_t, is_call,
                                            model="black_scholes")

    # 4) Backward — sum(σ).backward() yields ∂σ/∂price = 1/vega for each row.
    sigma_hat.sum().backward()

    # 5) Cross-check against analytic vega from the BSM kernel.
    from fast_vollib.backends.torch_backend import _price_vega_d1d2_t

    with torch.no_grad():
        _, vega, _, _ = _price_vega_d1d2_t(is_call, s_t, k_t, t_t, r_t, sigma_true, q_zero)
        analytic = 1.0 / vega

    rel_err = (price.grad - analytic).abs() / analytic.abs()
    print("∂σ/∂price = 1/vega   (verifying the implicit-function identity)")
    print()
    print(f"{'K':>6}  {'σ (recovered)':>15}  {'autograd':>10}  {'analytic':>10}  {'rel. err':>10}")
    print("-" * 65)
    for k, sg, ag, an, e in zip(k_t, sigma_hat, price.grad, analytic, rel_err):
        print(f"{k.item():>6.1f}  {sg.item():>15.10f}  {ag.item():>10.6f}  "
              f"{an.item():>10.6f}  {e.item():>10.2e}")

In [ ]:
"""implied_volatility_autograd_jax (JAX): same contract, custom_vjp backend."""
if not HAS_JAX:
    print("JAX not installed — skipping JAX differentiable-IV demo.")
else:
    import jax
    import jax.numpy as jnp

    from fast_vollib.jackel import implied_volatility_autograd_jax
    from fast_vollib.jackel.differentiable_jax import _bsm_price_j, _price_vega_d1d2_j

    n = 8
    s_j = jnp.full((n,), 100.0)
    k_j = jnp.linspace(85.0, 115.0, n)
    t_j = jnp.full((n,), 0.5)
    r_j = jnp.full((n,), 0.05)
    sigma_true_j = jnp.full((n,), 0.25)
    q_zero_j = jnp.zeros(n)
    is_call_j = jnp.ones(n, dtype=bool)

    price_j = _bsm_price_j(is_call_j, s_j, k_j, t_j, r_j, sigma_true_j, q_zero_j)

    # JIT-compiled grad-of-sum-of-σ wrt price → 1/vega per row
    @jax.jit
    def sum_iv(price):
        return implied_volatility_autograd_jax(
            price, s_j, k_j, t_j, r_j, is_call_j, q=q_zero_j, model="black_scholes"
        ).sum()

    grad_price = jax.jit(jax.grad(sum_iv))(price_j)

    # Analytic 1/vega
    _, vega_j, _, _ = _price_vega_d1d2_j(is_call_j, s_j, k_j, t_j, r_j, sigma_true_j, q_zero_j)
    analytic_j = 1.0 / vega_j

    rel = jnp.abs(grad_price - analytic_j) / jnp.abs(analytic_j)
    print("JAX:  ∂σ/∂price  vs  1/vega   (max rel. err: {:.2e})".format(float(jnp.max(rel))))
    print()
    print(f"{'K':>6}  {'autograd':>12}  {'analytic':>12}  {'rel. err':>10}")
    print("-" * 50)
    for k, ag, an, e in zip(k_j, grad_price, analytic_j, rel):
        print(f"{float(k):>6.1f}  {float(ag):>12.6f}  {float(an):>12.6f}  {float(e):>10.2e}")

---
## 9. Greeks

| Greek | Sensitivity to | Typical use |
|---|---|---|
| **Delta** (Δ) | Underlying price | Hedge ratio, directional exposure |
| **Gamma** (Γ) | Delta (second-order) | Convexity, re-hedge frequency |
| **Theta** (Θ) | Time decay (per day) | Daily P&L cost of holding |
| **Vega** (ν) | Implied vol (per 1%) | Vol trading, vega-hedging |
| **Rho** (ρ) | Interest rate (per 1%) | Rate sensitivity |

**Key optimization**: `get_all_greeks()` computes all 5 in a **single pass** — d₁, d₂, N(d₁), N(d₂) are evaluated once total, versus 5 redundant passes if you call each Greek individually.

In [ ]:
"""Individual Greeks — each with its own function."""
from fast_vollib import (
    vectorized_delta,
    vectorized_gamma,
    vectorized_rho,
    vectorized_theta,
    vectorized_vega,
)

flag_ = np.array(["c"])
S_ = np.array([450.0])
K_ = np.array([450.0])
t_ = np.array([0.25])
r_ = np.array([0.05])
sig_ = np.array([0.20])

delta = vectorized_delta(flag_, S_, K_, t_, r_, sig_, return_as="numpy")
gamma = vectorized_gamma(flag_, S_, K_, t_, r_, sig_, return_as="numpy")
theta = vectorized_theta(flag_, S_, K_, t_, r_, sig_, return_as="numpy")
vega = vectorized_vega(flag_, S_, K_, t_, r_, sig_, return_as="numpy")
rho = vectorized_rho(flag_, S_, K_, t_, r_, sig_, return_as="numpy")

print("ATM SPY Call  (S=K=$450, T=3m, σ=20%, r=5%)")
print("─" * 60)
print(f"  Δ Delta  : {delta[0]:>+9.4f}   hedge = {delta[0] * 100:.1f} shares per 100-lot")
print(f"  Γ Gamma  : {gamma[0]:>+9.4f}   Δ shifts {gamma[0]:.4f} per $1 move")
print(f"  Θ Theta  : {theta[0]:>+9.4f}   ${abs(theta[0] * 100):.2f}/day decay per 100 contracts")
print(f"  ν Vega   : {vega[0]:>+9.4f}   ${vega[0] * 100:.2f} per 1% vol rise, per 100 contracts")
print(f"  ρ Rho    : {rho[0]:>+9.4f}   ${abs(rho[0] * 100):.2f} per 1% rate move, per 100 contracts")

In [ ]:
"""get_all_greeks() — all 5 in one optimized pass, portfolio view."""
import pandas as pd

from fast_vollib import get_all_greeks

flag_arr = np.array(["c", "c", "p", "p", "c"])
S_arr = np.array([450.0, 450.0, 450.0, 450.0, 200.0])
K_arr = np.array([440.0, 460.0, 440.0, 460.0, 220.0])
t_arr = np.array([0.25, 0.25, 0.25, 0.25, 0.50])
r_arr = np.array([0.05, 0.05, 0.05, 0.05, 0.05])
sig_arr = np.array([0.20, 0.20, 0.20, 0.20, 0.65])
qty_arr = np.array([10, -5, -10, 5, 20])
descs = ["SPY ITM call", "SPY OTM call", "SPY OTM put", "SPY ITM put", "TSLA OTM call"]

# Single pass — 5x faster than calling each Greek separately
greeks_df = get_all_greeks(flag_arr, S_arr, K_arr, t_arr, r_arr, sig_arr, return_as="dataframe")

result = pd.concat(
    [pd.DataFrame({"description": descs, "qty": qty_arr}), greeks_df.reset_index(drop=True)], axis=1
)

print("Portfolio Greeks (per contract):")
print(result[["description", "qty", "delta", "gamma", "theta", "vega", "rho"]].to_string(index=False))

print("\nNet Portfolio Greeks (qty-weighted):")
for g in ["delta", "gamma", "theta", "vega", "rho"]:
    net = (result[g] * result["qty"]).sum()
    print(f"  {g.capitalize():<8}: {net:>+10.4f}")

---
## 10. Visualizing Greeks Profiles

Greeks vary with **moneyness** and **time to expiry**. The charts below show all 5 Greeks
simultaneously across a spot price range, for four expiries.

In [ ]:
"""Greeks as a function of spot price and time to expiry."""
if not HAS_MPL:
    print("Matplotlib not available — skipping Greeks visualization.")
else:
    import matplotlib.pyplot as plt

    from fast_vollib import get_all_greeks

    plt.rcParams.update({"font.size": 11, "axes.titlesize": 12})
    S_grid = np.linspace(370, 530, 200)
    K_fixed, r_fixed, sig_fixed = 450.0, 0.05, 0.20
    expiries = {"1 week": 1 / 52, "1 month": 1 / 12, "3 months": 0.25, "1 year": 1.0}
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    greek_names = ["delta", "gamma", "theta", "vega", "rho"]

    for idx, greek in enumerate(greek_names):
        ax = axes.flatten()[idx]
        for (lbl, tv), col in zip(expiries.items(), colors):
            n = len(S_grid)
            g = get_all_greeks(
                np.full(n, "c"),
                S_grid,
                np.full(n, K_fixed),
                np.full(n, tv),
                np.full(n, r_fixed),
                np.full(n, sig_fixed),
                return_as="dict",
            )
            ax.plot(S_grid, g[greek], label=lbl, color=col, linewidth=2)
        ax.axvline(K_fixed, color="gray", linestyle=":", linewidth=1)
        ax.set_title(greek.capitalize())
        ax.set_xlabel("Spot Price (S)")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

    # 6th panel: Gamma vs Delta (convexity profile)
    ax = axes.flatten()[5]
    for (lbl, tv), col in zip(expiries.items(), colors):
        n = len(S_grid)
        g = get_all_greeks(
            np.full(n, "c"),
            S_grid,
            np.full(n, K_fixed),
            np.full(n, tv),
            np.full(n, r_fixed),
            np.full(n, sig_fixed),
            return_as="dict",
        )
        ax.plot(g["delta"], g["gamma"], label=lbl, color=col, linewidth=2)
    ax.set_xlabel("Delta")
    ax.set_ylabel("Gamma")
    ax.set_title("Gamma vs Delta  (convexity)")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    fig.suptitle(
        f"European Call Greeks  (K=${K_fixed:.0f}, σ={sig_fixed * 100:.0f}%, r={r_fixed * 100:.0f}%)",
        fontsize=14,
        fontweight="bold",
        y=1.01,
    )
    plt.tight_layout()
    plt.show()

---
## 11. DataFrame API

`price_dataframe()` enriches any Pandas DataFrame of option records end-to-end.

- **σ → Price + Greeks** — given implied vol, compute price and all Greeks
- **Price → IV + Greeks** — given market price, compute implied vol and all Greeks

> The `Price → IV` path internally uses the legacy Halley solver. To enrich a DataFrame
> with **Jäckel-quality** IV instead, compute IV via `jackel_iv()` and then call
> `price_dataframe(..., sigma_col="sigma")` to get Greeks. The cell below demonstrates both.

> `inplace=False` returns a **new** DataFrame with only the computed columns (`Price`, `delta`, …).
> Concatenate it with your input to keep the full picture.

In [ ]:
"""price_dataframe: enrich a DataFrame of options with prices and Greeks."""
import pandas as pd

from fast_vollib import price_dataframe

book = pd.DataFrame(
    {
        "flag": ["c", "c", "c", "p", "p", "p"],
        "S": [450, 450, 450, 450, 450, 450],
        "K": [430, 450, 470, 430, 450, 470],
        "t": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25],
        "r": [0.05, 0.05, 0.05, 0.05, 0.05, 0.05],
        "sigma": [0.21, 0.20, 0.19, 0.21, 0.20, 0.19],  # slight vol smile
        "position": [10, -20, 15, 15, -20, 10],
    }
)

# inplace=False → returns only the computed columns; merge back for full view
computed = price_dataframe(
    book,
    flag_col="flag",
    underlying_price_col="S",
    strike_col="K",
    annualized_tte_col="t",
    riskfree_rate_col="r",
    sigma_col="sigma",
    inplace=False,
)
enriched = pd.concat([book.reset_index(drop=True), computed.reset_index(drop=True)], axis=1)

print("Option Book — Prices and Greeks")
cols = ["flag", "K", "sigma", "Price", "delta", "gamma", "theta", "vega"]
print(enriched[cols].to_string(index=False))

print("\nNet Portfolio Greeks (qty-weighted):")
for g in ["Price", "delta", "gamma", "theta", "vega"]:
    net = (enriched[g] * enriched["position"]).sum()
    print(f"  {g:<8}: {net:>+10.4f}")

In [ ]:
"""Price → Jäckel IV → Greeks: a two-step DataFrame pipeline."""
import pandas as pd

from fast_vollib import price_dataframe

# Same chain as above, but now expressed via market prices instead of σ
mkt = pd.DataFrame(
    {
        "flag": ["c", "c", "c", "p", "p", "p"],
        "S": [450, 450, 450, 450, 450, 450],
        "K": [430, 450, 470, 430, 450, 470],
        "t": [0.25, 0.25, 0.25, 0.25, 0.25, 0.25],
        "r": [0.05, 0.05, 0.05, 0.05, 0.05, 0.05],
        "MarketPrice": enriched["Price"].to_numpy(),  # observed market price
    }
)

# Step 1: invert price → σ via the Jäckel solver
mkt["sigma"] = jackel_iv(
    mkt["MarketPrice"].to_numpy(),
    mkt["S"].to_numpy(dtype=float),
    mkt["K"].to_numpy(dtype=float),
    mkt["t"].to_numpy(dtype=float),
    mkt["r"].to_numpy(dtype=float),
    mkt["flag"].to_numpy(),
)

# Step 2: σ → Greeks via price_dataframe (Price column here is the model
# re-price under recovered σ — should match MarketPrice to ~1e-9).
greeks_only = price_dataframe(
    mkt,
    flag_col="flag",
    underlying_price_col="S",
    strike_col="K",
    annualized_tte_col="t",
    riskfree_rate_col="r",
    sigma_col="sigma",
    inplace=False,
)
combined = pd.concat([mkt.reset_index(drop=True), greeks_only.reset_index(drop=True)], axis=1)

print("Market price → Jäckel σ → re-priced Price + Greeks")
print(combined[["flag", "K", "MarketPrice", "sigma", "Price",
                "delta", "gamma", "theta", "vega"]].round(6).to_string(index=False))

reprice_err = np.max(np.abs(combined["Price"] - combined["MarketPrice"]))
print(f"\nMax |re-price − market| across the chain: {reprice_err:.2e}")

---
## 12. Building an Option Chain

A full option chain — all strikes, both sides — is a natural vectorization target.

In [ ]:
"""Build a complete SPY option chain with vol smile, prices and Greeks."""
import pandas as pd

from fast_vollib import price_dataframe

S_spot = 450.0
strikes = np.arange(380, 521, 5, dtype=float)  # $380–$520 in $5 steps
N_strikes = len(strikes)

# Realistic equity vol smile: ATM=20%, OTM wings bid up
log_m = np.log(strikes / S_spot)
vol_smile = 0.20 + 0.05 * log_m**2 - 0.03 * log_m

# Price calls and puts separately (keeps flag accessible for filtering)
calls_in = pd.DataFrame(
    {"flag": "c", "S": S_spot, "K": strikes, "t": 0.25, "r": 0.05, "sigma": vol_smile}
)
puts_in = calls_in.copy()
puts_in["flag"] = "p"


def _enrich(df):
    comp = price_dataframe(
        df,
        flag_col="flag",
        underlying_price_col="S",
        strike_col="K",
        annualized_tte_col="t",
        riskfree_rate_col="r",
        sigma_col="sigma",
        inplace=False,
    )
    return pd.concat([df.reset_index(drop=True), comp.reset_index(drop=True)], axis=1)


calls_out = _enrich(calls_in)
puts_out = _enrich(puts_in)

print(f"Chain: {N_strikes} strikes × 2 = {2 * N_strikes} contracts priced & Greek-ified")
print()

# Display ATM ±4 strikes
atm_idx = int(np.argmin(np.abs(strikes - S_spot)))
lo, hi = max(0, atm_idx - 4), min(N_strikes, atm_idx + 5)
display = pd.DataFrame(
    {
        "Call Δ": calls_out["delta"].iloc[lo:hi].round(3).values,
        "Call Price": calls_out["Price"].iloc[lo:hi].round(3).values,
        "Strike": strikes[lo:hi],
        "Put Price": puts_out["Price"].iloc[lo:hi].round(3).values,
        "Put Δ": puts_out["delta"].iloc[lo:hi].round(3).values,
    }
)
print("ATM-centred chain (±4 strikes):")
print(display.to_string(index=False))

if HAS_MPL:
    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(calls_out["K"], calls_out["Price"], "b-", lw=2, label="Call")
    ax1.plot(puts_out["K"], puts_out["Price"], "r--", lw=2, label="Put")
    ax1.axvline(S_spot, color="gray", ls=":", lw=1, label=f"S=${S_spot:.0f}")
    ax1.set_xlabel("Strike")
    ax1.set_ylabel("Option Price ($)")
    ax1.set_title("SPY Option Chain — Prices (3m)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(calls_out["K"], calls_out["delta"], "b-", lw=2, label="Call Δ")
    ax2.plot(puts_out["K"], puts_out["delta"], "r--", lw=2, label="Put Δ")
    ax2.axvline(S_spot, color="gray", ls=":", lw=1)
    ax2.axhline(0, color="black", lw=0.5)
    ax2.set_xlabel("Strike")
    ax2.set_ylabel("Delta")
    ax2.set_title("SPY Option Chain — Delta (3m)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## 13. Broadcasting & Vectorization

All inputs broadcast à la NumPy. Compute a full **price matrix** over a (σ × strike) grid
with a single vectorized call — no Python loops.

In [ ]:
"""Price matrix over a (sigma × strike) grid in one call."""
import pandas as pd

from fast_vollib import fast_black_scholes

S0 = 100.0
sigmas_1d = np.array([0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50])
strikes_1d = np.array([85.0, 90.0, 95.0, 100.0, 105.0, 110.0, 115.0])

sig_grid, K_grid = np.meshgrid(sigmas_1d, strikes_1d, indexing="ij")
n_total = sig_grid.size

prices = fast_black_scholes(
    flag=np.full(n_total, "c"),
    S=np.full(n_total, S0),
    K=K_grid.ravel(),
    t=np.full(n_total, 0.25),
    r=np.full(n_total, 0.05),
    sigma=sig_grid.ravel(),
    return_as="numpy",
).reshape(sig_grid.shape)

price_table = pd.DataFrame(
    prices, index=pd.Index(sigmas_1d, name="vol↓ / strike→"), columns=strikes_1d
).round(3)

print(f"Call prices: {n_total} contracts in one vectorized call  (S=$100, T=3m, r=5%)")
print()
print(price_table.to_string())

---
## 14. Volatility Surface

A full IV surface requires computing thousands of IV solves across all (strike, expiry) pairs.
With the Jäckel solver this round-trips to **machine precision** — the recovered surface differs
from the parametric model only by float64 noise.

In [ ]:
"""Build a full implied volatility surface from a parametric model."""
from fast_vollib import fast_black_scholes

S0 = 4500.0
expiries_dte = np.array([7, 14, 30, 60, 91, 120, 180, 252])
expiries_ann = expiries_dte / 252.0
moneyness_grid = np.linspace(0.70, 1.30, 61)
strikes_surf = moneyness_grid * S0


def parametric_vol(m, T):
    """ATM vol term structure + equity skew + smile curvature."""
    x = np.log(m)
    return np.clip(0.18 + 0.04 * np.sqrt(T) - 0.12 * x / np.sqrt(T) + 0.20 * x**2 / T, 0.05, 2.0)


n_exp, n_k = len(expiries_ann), len(strikes_surf)
iv_surface = np.zeros((n_exp, n_k))

for i, t_ann in enumerate(expiries_ann):
    true_vol = parametric_vol(moneyness_grid, t_ann)
    mkt_prices = fast_black_scholes(
        np.full(n_k, "c"),
        np.full(n_k, S0),
        strikes_surf,
        np.full(n_k, t_ann),
        np.full(n_k, 0.05),
        true_vol,
        return_as="numpy",
    )
    iv_surface[i] = jackel_iv(
        mkt_prices,
        np.full(n_k, S0),
        strikes_surf,
        np.full(n_k, t_ann),
        np.full(n_k, 0.05),
        np.full(n_k, "c"),
    )

total = n_exp * n_k
true_surf = np.array([parametric_vol(moneyness_grid, t) for t in expiries_ann])
valid = ~np.isnan(iv_surface)
max_err = np.max(np.abs(iv_surface[valid] - true_surf[valid]))
print(f"IV surface: {n_exp} expiries × {n_k} strikes = {total:,} solves")
print(f"Max round-trip error: {max_err:.2e}  (target < 1e-9, Jäckel guarantee ~2e-11)")

if HAS_MPL:
    from matplotlib import cm
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    T_grid, M_grid = np.meshgrid(expiries_dte, moneyness_grid, indexing="ij")
    fig = plt.figure(figsize=(14, 6))

    ax3d = fig.add_subplot(121, projection="3d")
    surf = ax3d.plot_surface(
        M_grid, T_grid, iv_surface * 100, cmap=cm.RdYlGn_r, alpha=0.85, linewidth=0
    )
    ax3d.set_xlabel("Moneyness", labelpad=8)
    ax3d.set_ylabel("Expiry (DTE)", labelpad=8)
    ax3d.set_zlabel("IV (%)", labelpad=6)
    ax3d.set_title("Implied Volatility Surface", fontweight="bold")
    fig.colorbar(surf, ax=ax3d, shrink=0.5, aspect=10, label="IV (%)")

    ax2d = fig.add_subplot(122)
    cmap = plt.cm.plasma(np.linspace(0, 0.85, n_exp))
    for i, (dte, col) in enumerate(zip(expiries_dte, cmap)):
        ax2d.plot(moneyness_grid, iv_surface[i] * 100, color=col, lw=2, label=f"{dte}d")
    ax2d.axvline(1.0, color="gray", ls=":", lw=1)
    ax2d.set_xlabel("Moneyness (K/S)")
    ax2d.set_ylabel("IV (%)")
    ax2d.set_title("Vol Smiles by Expiry", fontweight="bold")
    ax2d.legend(fontsize=8, ncol=2)
    ax2d.grid(True, alpha=0.3)

    plt.suptitle(
        "SPX Implied Volatility Surface  (parametric model · Jäckel solver)",
        fontsize=13, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

---
## 15. Backend Performance Comparison

Throughput across the three Jäckel backends. The CPU path is Numba-JIT-fused; the PyTorch
path uses `torch.compile(dynamic=True)`; the JAX path uses `jax.jit` + `lax.fori_loop`. On a
warm GPU, expect order-of-magnitude speedups vs the CPU path at large N.

> **Pricing** (Black-Scholes) routes through `fast_vollib.backends.<backend>` (NumPy / Triton /
> JAX); these numbers are reported separately from the IV-solver figures.

In [ ]:
"""Pricing & Jäckel-IV throughput benchmark."""
import time
import warnings

from fast_vollib import fast_black_scholes


def _make_inputs(n, seed=42):
    rng = np.random.default_rng(seed)
    return dict(
        flag=np.where(rng.random(n) > 0.5, "c", "p"),
        S=rng.uniform(80, 120, n),
        K=rng.uniform(75, 125, n),
        t=rng.uniform(1 / 52, 2.0, n),
        r=rng.uniform(0.01, 0.08, n),
        sigma=rng.uniform(0.10, 0.80, n),
    )


def _bench(fn, kwargs, n_runs=3):
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        fn(**kwargs)
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


batch_sizes = [1_000, 10_000, 100_000, 1_000_000]
backends_to_bench = ["numpy"] + (["torch"] if HAS_TORCH else [])
iv_backends = ["numpy"] + (["torch"] if HAS_TORCH else []) + (["jax"] if HAS_JAX else [])


def _print_bench(title, results, backends, *, speedup_against=None):
    """Print a benchmark table; ``speedup_against`` names the backend to compare."""
    print(title)
    print("-" * 70)
    hdr = f"{'N':>10}"
    for b in backends:
        hdr += f"  {b:>14}"
    if speedup_against and speedup_against in backends and len(backends) > 1:
        hdr += f"  vs-{speedup_against}"
    print(hdr)
    print("-" * 70)
    for n, row_times in results:
        row = f"{n:>10,}"
        for b in backends:
            row += f"  {n / row_times[b] / 1e6:>10.2f} M/s"
        if speedup_against and speedup_against in backends and len(backends) > 1:
            best = min(row_times.values())
            row += f"  {row_times[speedup_against] / best:>5.2f}x"
        print(row)


warnings.filterwarnings("ignore")

# ── Pricing: NumPy vs Torch (Halley/jackel sharing same pricing path) ────
pricing_results = []
for n in batch_sizes:
    inp = _make_inputs(n)
    row = {}
    for b in backends_to_bench:
        row[b] = _bench(fast_black_scholes, {**inp, "return_as": "numpy", "backend": b})
    pricing_results.append((n, row))

_print_bench("Pricing throughput (Black-Scholes)", pricing_results, backends_to_bench,
             speedup_against="numpy")

# ── Jäckel IV: NumPy / Torch / JAX ──────────────────────────────────────
print()
iv_results = []
for n in batch_sizes:
    inp = _make_inputs(n)
    prices = fast_black_scholes(**inp, return_as="numpy", backend="numpy")
    row = {}
    for b in iv_backends:
        # Warm-up for compiled backends
        if b in {"torch", "jax"}:
            _ = jackel_iv(prices[:1024], inp["S"][:1024], inp["K"][:1024],
                          inp["t"][:1024], inp["r"][:1024], inp["flag"][:1024], backend=b)
        row[b] = _bench(
            jackel_iv,
            dict(price=prices, S=inp["S"], K=inp["K"], t=inp["t"], r=inp["r"],
                 flag=inp["flag"], backend=b),
        )
    iv_results.append((n, row))

_print_bench("Jäckel IV throughput (Black-Scholes)", iv_results, iv_backends,
             speedup_against="numpy")
warnings.filterwarnings("default")

print()
if CUDA_AVAILABLE:
    print("GPU detected — for max throughput at N ≥ 100k see the Triton kernel section below.")
elif HAS_TORCH:
    print("CPU PyTorch: torch.compile() overhead dominates at small N.")
    print("Expect order-of-magnitude speedup on GPU (Triton) at N ≥ 100k.")

---
## 16. Triton — Maximum-Throughput GPU IV

The single fastest IV path in fast-vollib is the **single-pass Triton kernel** in
`fast_vollib.jackel.triton_kernels.jackel_iv_triton`. The full algorithm — preproc, boundary,
Hermite guess, and three Householder(3) iterations — runs in **one** `@triton.jit` kernel
with all intermediate values living in registers (no HBM round-trips).

| Path | GPU compute / 100k options (H100 NVL) |
|---|---|
| `jackel.torch_backend` (`torch.compile`) | ~2.7 ms |
| `jackel.jax_backend` (`@jax.jit`) | ~2.4 ms |
| **`jackel.triton_kernels.jackel_iv_triton`** | **~0.056 ms** (≈ 11× faster) |

The kernel works in undiscounted Black-76 space — pass `F` (forward) and an undiscounted
price. For Black-Scholes / BSM, multiply your discounted price by `exp(rT)` and use
`F = S·exp((r−q)T)`.

> Triton is only available on CUDA. The cell below skips itself if `HAS_TRITON` is False.

In [ ]:
"""jackel_iv_triton: the fastest IV path on CUDA."""
if not HAS_TRITON:
    print("Triton not available (requires CUDA + `pip install triton`) — skipping.")
else:
    import torch

    from fast_vollib.jackel.triton_kernels import jackel_iv_triton
    from fast_vollib.jackel.jackel_iv import jackel_iv_black

    rng = np.random.default_rng(0)
    n = 100_000
    F = 100.0
    T = 0.5
    sigma_true = rng.uniform(0.10, 0.60, n)
    K_np = F * np.exp(rng.uniform(-0.30, 0.30, n))
    is_call_np = rng.random(n) > 0.5

    # Reference prices via the analytic normalised_black_call
    from fast_vollib.jackel.jackel_iv import normalised_black_call
    sqrt_FK = np.sqrt(F * K_np)
    x = np.log(F / K_np)
    s = sigma_true * np.sqrt(T)            # already array-shaped
    beta_call = normalised_black_call(x, s)
    beta_put = normalised_black_call(-x, s)
    price_np = sqrt_FK * np.where(is_call_np, beta_call, beta_put)

    # GPU buffers
    dev = torch.device("cuda")
    price_t = torch.as_tensor(price_np, dtype=torch.float64, device=dev)
    K_t = torch.as_tensor(K_np, dtype=torch.float64, device=dev)
    is_call_t = torch.as_tensor(is_call_np, dtype=torch.bool, device=dev)

    # Warm-up
    _ = jackel_iv_triton(price_t[:1024], F, K_t[:1024], T, is_call_t[:1024])
    torch.cuda.synchronize()

    # Time the kernel with CUDA events
    ev0 = torch.cuda.Event(enable_timing=True)
    ev1 = torch.cuda.Event(enable_timing=True)
    n_rounds = 10
    ev0.record()
    for _ in range(n_rounds):
        sigma_triton = jackel_iv_triton(price_t, F, K_t, T, is_call_t)
    ev1.record()
    torch.cuda.synchronize()
    ms_total = ev0.elapsed_time(ev1)
    print(f"Triton: {ms_total / n_rounds:.3f} ms / {n:,} options")

    # Accuracy vs the NumPy reference
    sigma_ref = jackel_iv_black(price_np, F, K_np, T, is_call=is_call_np)
    sigma_triton_np = sigma_triton.cpu().numpy()
    valid = ~np.isnan(sigma_ref) & ~np.isnan(sigma_triton_np) & (sigma_ref > 0)
    rel_err = np.abs(sigma_triton_np[valid] - sigma_ref[valid]) / sigma_ref[valid]
    print(f"Max relative error vs NumPy reference: {np.max(rel_err):.2e}")

---
## 17. py_vollib Drop-in Compatibility

If your codebase uses [`py_vollib`](https://github.com/vollib/py_vollib), you can accelerate it
**without changing any call sites** by calling `patch_py_vollib()` once at startup. The patched
implementations route pricing through fast-vollib (and IV through the legacy Halley solver in
the compat layer — the Jäckel solver is invoked via the new `fast_vollib.jackel.*` API).

```python
from fast_vollib.compat import patch_py_vollib
patch_py_vollib()   # all subsequent py_vollib calls route through fast-vollib
```

In [ ]:
"""py_vollib drop-in compatibility demo."""
HAS_PY_VOLLIB = importlib.util.find_spec("py_vollib") is not None

if not HAS_PY_VOLLIB:
    print("py_vollib not installed — showing interface only.")
    print()
    print("Usage:")
    print("  pip install py_vollib")
    print("  from fast_vollib.compat import patch_py_vollib")
    print("  patch_py_vollib()  # done — zero other code changes needed")
    print()
    print("Modules patched transparently:")
    for m in [
        "py_vollib.black",
        "py_vollib.black_scholes",
        "py_vollib.black_scholes_merton",
        "py_vollib.black.implied_volatility",
        "py_vollib.black_scholes.implied_volatility",
        "py_vollib.black_scholes_merton.implied_volatility",
        "py_vollib.black.greeks.analytical",
        "py_vollib.black_scholes.greeks.analytical",
        "py_vollib.black_scholes_merton.greeks.analytical",
    ]:
        print(f"  ✓ {m}")
else:
    import importlib as _il

    import py_vollib.black_scholes

    _il.reload(py_vollib.black_scholes)  # reset to unpatched state (makes cell re-runnable)
    import py_vollib.black_scholes as pv_bs

    from fast_vollib.compat import patch_py_vollib

    args = ("c", 100.0, 100.0, 0.25, 0.05, 0.20)

    price_before = pv_bs.black_scholes(*args)
    print(f"Before patch: py_vollib price = {price_before:.6f}")

    patch_py_vollib()

    price_after = float(pv_bs.black_scholes(*args).squeeze())
    print(f"After patch : fast-vollib     = {price_after:.6f}")
    print(f"Prices match: {abs(price_before - price_after) < 1e-6}")
    print("\nExisting py_vollib code now runs on fast-vollib with zero changes.")

---
## 18. Advanced Patterns

### `return_native`: staying on the GPU

Use `return_native=True` to avoid the device→host copy when the result stays in a GPU pipeline.

In [ ]:
"""return_native=True: keep tensors on GPU — no PCIe round-trip."""
if not HAS_TORCH:
    print("PyTorch not installed — skipping.")
else:
    import torch

    from fast_vollib import fast_black_scholes

    n = 1_000
    rng = np.random.default_rng(0)
    fn = np.where(rng.random(n) > 0.5, "c", "p")
    St = torch.as_tensor(rng.uniform(90, 110, n), dtype=torch.float64)
    Kt = torch.as_tensor(rng.uniform(85, 115, n), dtype=torch.float64)
    tt = torch.as_tensor(rng.uniform(0.05, 1.0, n), dtype=torch.float64)
    rt = torch.as_tensor(rng.uniform(0.02, 0.07, n), dtype=torch.float64)
    st = torch.as_tensor(rng.uniform(0.10, 0.60, n), dtype=torch.float64)

    prices_t = fast_black_scholes(
        fn, St, Kt, tt, rt, st,
        backend="torch",
        return_native=True,  # ← stays on device
    )

    print(f"Output type   : {type(prices_t).__name__}")
    print(f"Output device : {prices_t.device}")
    print(f"Output dtype  : {prices_t.dtype}")
    print(f"Output shape  : {prices_t.shape}")
    print(f"Sample prices : {prices_t[:5].cpu().numpy().round(4)}")
    print()
    print("Use case: embed fast-vollib in a PyTorch model or loss function — pair this with")
    print("`implied_volatility_autograd` (Section 8) to get end-to-end differentiable IV.")

In [ ]:
"""Environment variable backend control — useful for CI/CD and A/B testing."""
import os

from fast_vollib.config import get_backend

os.environ["FAST_VOLLIB_BACKEND"] = "numpy"
print(f"FAST_VOLLIB_BACKEND=numpy  → get_backend(): {get_backend()}")

del os.environ["FAST_VOLLIB_BACKEND"]

print(f"(no env var)               → get_backend(): {get_backend()}  [auto-detect]")

print("\nPer-call kwarg always wins over env var:")
from fast_vollib import fast_black_scholes

os.environ["FAST_VOLLIB_BACKEND"] = "torch" if HAS_TORCH else "numpy"
result = fast_black_scholes(
    np.array(["c"]),
    np.array([100.0]),
    np.array([100.0]),
    np.array([0.25]),
    np.array([0.05]),
    np.array([0.20]),
    backend="numpy",  # explicit override
    return_as="numpy",
)
print(f"  backend='numpy' (explicit) → {result[0]:.6f}")
del os.environ["FAST_VOLLIB_BACKEND"]

---
## 19. Summary & Next Steps

### What we covered

| Topic | Key function(s) |
|---|---|
| Platform detection | `available_backends()`, `get_backend()` |
| Backend configuration | `set_backend()`, env vars, per-call kwarg |
| Black-Scholes pricing | `fast_black_scholes()` |
| Black-76 pricing | `fast_black()` |
| BSM (dividends/FX) | `fast_black_scholes_merton()` |
| **Jäckel implied volatility** | `fast_vollib.jackel.numpy_backend.implied_volatility` (`jackel_iv`) |
| **Direct Black-76 IV** | `fast_vollib.jackel.jackel_iv.jackel_iv_black` |
| **Differentiable IV (PyTorch)** | `fast_vollib.jackel.implied_volatility_autograd` |
| **Differentiable IV (JAX)** | `fast_vollib.jackel.implied_volatility_autograd_jax` |
| **Triton GPU kernel** | `fast_vollib.jackel.triton_kernels.jackel_iv_triton` |
| Individual Greeks | `vectorized_delta/gamma/theta/vega/rho()` |
| All Greeks (fast) | `get_all_greeks()` — single-pass |
| DataFrame enrichment | `price_dataframe()` |
| Vectorization | Broadcasting over arbitrary grids |
| Volatility surface | Full (expiry × strike) IV surface |
| Performance | Backend throughput benchmarks |
| GPU tensors | `return_native=True` |
| Legacy migration | `patch_py_vollib()` |

---

### Performance tips

1. **Use `get_all_greeks()`** instead of calling each Greek separately — 5× fewer array passes.
2. **Use `return_as='numpy'`** when feeding into further NumPy computation.
3. **Use `return_native=True`** with `backend='torch'` when the result stays on GPU.
4. **For machine-precision IV**, call the Jäckel solver directly (`jackel_iv` wrapper, or `jackel_iv_black` for Black-76) — the legacy `fast_implied_volatility` Halley path is faster on small batches but only ~10⁻⁸ accurate.
5. **For maximum GPU throughput**, use `jackel_iv_triton` — single-pass, ~0.056 ms / 100k options on H100 NVL.
6. **For training loops**, use `implied_volatility_autograd` (PyTorch) or `implied_volatility_autograd_jax` — exact gradients via the implicit-function theorem.
7. **Install Numba** (`pip install numba`) on CPU — the Jäckel CPU path uses Numba kernels for preproc / postproc.

---

### Where to go next

- **Documentation**: <https://raeidsaqur.github.io/fast-vollib/>
- **Jäckel solver guide**: `docs/jackel.md`
- **Differentiable IV guide**: `docs/differentiable_iv.md`
- **API reference**: `docs/api.md`
- **Backend guide**: `docs/backends.md`
- **py_vollib migration**: `docs/compatibility.md`
- **GitHub**: <https://github.com/raeidsaqur/fast-vollib>

---

*Built with care for quantitative finance — fast-vollib · Jäckel "Let's Be Rational" edition*